Chapters 7 and 8 cover Experiment Analysis (A/B Testing), Complex Joins, CTEs, and Query Optimization.

These final chapters pull together everything from previous chapters into modular, production-grade SQL architectures. For a quantitative analyst or data scientist, this is where SQL transitions from single-query data extraction to building repeatable analytical pipelines and experimental evaluation frameworks inside database clusters.

The 4 Core Takeaways of Chapters 7 & 8

1. A/B Testing & Experimentation Frameworks in SQLEvaluating experiments (A/B testing, marketing campaigns, risk policy modifications) requires joining assignment logs with downstream user metrics across identical post-exposure time windows.

The Exposure Window Pattern: Align metrics to $t_0 = \text{exposure\_time}$ for each entity independently, ensuring that control and variant groups are measured over equivalent post-assignment durations (e.g., $t_0 \text{ to } t_0 + 14 \text{ days}$).

Conversion & Metric Aggregation: Computing baseline conversion rates, average revenue per user (ARPU), and variance metrics directly in SQL to feed standard hypothesis tests ($z$-tests, $t$-tests).

2. Complex Joins & Fan-Out PreventionJoining tables across different levels of aggregation (e.g., joining 1-to-many transactions to 1-to-many session logs) causes join fan-out, which duplicates rows and silently distorts sums, counts, and averages.

The Solution: Aggregate datasets to matching grain sizes inside separate Common Table Expressions (CTEs) before executing the join. Never perform SUM() or COUNT() across a multi-table join with multiple 1-to-many relationships.

3. Modular Pipeline Design with CTEs (WITH Clauses)Instead of writing monolithic queries filled with deeply nested subqueries, professional SQL uses chained Common Table Expressions (CTEs).

Design Philosophy: Build queries as an acyclic execution graph (DAG):$$\text{Raw Event Tables} \longrightarrow \text{Filtering/Cleaning CTE} \longrightarrow \text{Aggregation CTE} \longrightarrow \text{Final Output/Pivot}$$

Readability & Debugging: CTEs allow you to isolate logic, inspect intermediate step outputs, and maintain modular code.

4. Query Execution Plans & Performance OptimizationUnderstanding how database query engines parse and execute set transformations prevents out-of-memory errors and expensive full-table scans when scaling to large datasets.

Filter Pushdown: Place WHERE clauses as early as possible (inside initial CTEs) to reduce row cardinalities before executing joins or window functions.

Avoid Unnecessary Sorting: ORDER BY operations force a full data shuffle/sort. Omit ORDER BY from intermediate CTEs; apply sorting only in the final execution step.

Index & Column Selection: Select only required explicit columns (SELECT col1, col2) rather than wildcard scans (SELECT \*), allowing columnar engines like DuckDB to scan only targeted memory chunks.


In [ ]:
import duckdb
import pandas as pd

conn = duckdb.connect(database=':memory:')

# 1. Setup Synthetic A/B Testing Assignment Logs & Purchases
conn.execute("""
CREATE TABLE experiment_assignments (
    user_id INT,
    variant_group VARCHAR, -- 'control' vs 'treatment'
    assigned_at TIMESTAMP
);

CREATE TABLE user_purchases (
    user_id INT,
    purchase_amount NUMERIC(10,2),
    purchased_at TIMESTAMP
);

-- Seed Variant Assignments
INSERT INTO experiment_assignments VALUES
    (1, 'control',   '2026-03-01 10:00:00'),
    (2, 'control',   '2026-03-01 10:30:00'),
    (3, 'treatment', '2026-03-01 11:00:00'),
    (4, 'treatment', '2026-03-01 11:15:00');

-- Seed Purchases (Some before assignment, some post-assignment exposure window)
INSERT INTO user_purchases VALUES
    (1, 50.00, '2026-03-02 12:00:00'), -- Post-assignment
    (2, 20.00, '2026-02-28 09:00:00'), -- Pre-assignment (Ignore for AB test)
    (3, 85.00, '2026-03-01 14:00:00'), -- Post-assignment
    (3, 40.00, '2026-03-03 16:00:00'), -- Post-assignment
    (4, 110.00,'2026-03-02 10:00:00'); -- Post-assignment
""")

# 2. Chapters 7 & 8 Modular A/B Test Pipeline Query
ab_test_summary_df = conn.execute("""
WITH pre_filtered_assignments AS (
    -- Step 1: Filter and aggregate variant populations (Avoids Fan-Out)
    SELECT
        user_id,
        variant_group,
        assigned_at
    FROM experiment_assignments
),
post_exposure_purchases AS (
    -- Step 2: Compute conversions ONLY occurring within 14 days POST assignment
    SELECT
        a.user_id,
        a.variant_group,
        COALESCE(SUM(p.purchase_amount), 0.00) AS post_exposure_revenue,
        COUNT(p.purchase_amount) AS transaction_count
    FROM pre_filtered_assignments a
    LEFT JOIN user_purchases p
        ON a.user_id = p.user_id
       AND p.purchased_at >= a.assigned_at
       AND p.purchased_at <= a.assigned_at + INTERVAL 14 DAY
    GROUP BY a.user_id, a.variant_group
)
-- Step 3: Final Experiment Metrics (Conversion Rate, ARPU, Sample Size)
SELECT
    variant_group,
    COUNT(user_id) AS total_participants,
    COUNT(CASE WHEN post_exposure_revenue > 0 THEN 1 END) AS converted_users,

    -- Conversion Rate %
    ROUND(COUNT(CASE WHEN post_exposure_revenue > 0 THEN 1 END) * 100.0 / COUNT(user_id), 2) AS conversion_rate_pct,

    -- Average Revenue Per User (ARPU)
    ROUND(AVG(post_exposure_revenue), 2) AS arpu,

    -- Total Revenue Generated
    SUM(post_exposure_revenue) AS total_revenue
FROM post_exposure_purchases
GROUP BY variant_group
ORDER BY variant_group;
""").df()

ab_test_summary_df

| Concept                           | Problem Solved                                        | Primary SQL Technique                                        |
| :-------------------------------- | :---------------------------------------------------- | :----------------------------------------------------------- |
| **A/B Experiment Isolation**      | Measuring conversions strictly post-exposure          | `LEFT JOIN` on `user_id AND event_time >= assignment_time`   |
| **Join Fan-Out Prevention**       | Duplicating records when joining two 1-to-many tables | Aggregate tables in separate CTEs before joining             |
| **Modular Pipeline Architecture** | Spaghetti code and unreadable nested subqueries       | Multi-step `WITH cte1 AS (...), cte2 AS (...)` chains        |
| **Query Optimization**            | Slow query execution and memory spikes                | Filter early (`WHERE` pushdown), drop unnecessary `ORDER BY` |
